*Karen: **[Modeling owner]**: extracted from `airbnb_pricing_crispdm_v7.ipynb` for individual CRISP-DM phase attribution. Run from the repo root (same folder as `db.py`, `pricing_model.py`, etc.).*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install xgboost --quiet
from pricing_model import train_all_markets
results = train_all_markets()



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Bangkok         (THB): MAE=   574.27  R2=-0.014  n=392
Cape Town       (ZAR): MAE=  1213.27  R2=0.223  n=395
Hong Kong       (HKD): MAE=   170.46  R2=0.560  n=392
Istanbul        (TRY): MAE=   142.37  R2=0.045  n=393
Mexico City     (MXN): MAE=   436.89  R2=0.335  n=392
New York        (USD): MAE=    48.45  R2=0.091  n=393
Paris           (EUR): MAE=    38.23  R2=0.289  n=394
Rio de Janeiro  (BRL): MAE=   372.03  R2=0.096  n=393
Rome            (EUR): MAE=    34.15  R2=0.276  n=392
Sydney          (AUD): MAE=    64.49  R2=0.650  n=392


## 4. Modeling

Three model families — `RandomForestRegressor`, `GradientBoostingRegressor`, and `XGBRegressor` — are
tuned **independently for each market** via `RandomizedSearchCV`, not one fixed algorithm across all 10
markets. Whichever family scores best on cross-validation R² (train-split only) is what actually gets
saved and served for that market. This replaced the original single-Random-Forest-everywhere approach
after a notebook comparison (see Evaluation, below) showed Random Forest wasn't uniformly best — it
won only 2 of 10 markets when tested against the alternatives.

The feature set is unchanged: `accommodates`, `bedrooms`, `bathrooms` (approximated from `bedrooms`),
`dist_to_center_km`, `host_is_superhost`, `review_scores_rating`, `num_reviews`, `property_type`,
log-transformed `minimum_nights`/`maximum_nights`, `instant_bookable`, `host_response_rate`,
`host_acceptance_rate`, `host_total_listings_count`, all six review sub-scores, `amenities_count`, and
`neighbourhood` (K-fold target-encoded, out-of-fold on the training split only, so a listing's own
price never leaks into its own encoded feature). `room_type` and `property_type` are one-hot encoded,
scoped to each market's own categories.

Model selection uses each candidate's **CV score**, not the held-out test set — choosing a model family
based on test performance would leak test information into the decision, the same double-dipping the
project already avoids for hyperparameter tuning. The test set is touched exactly once, after the
winner is picked, purely to report honest final metrics.


## 5. Evaluation

Each per-market model is evaluated on a held-out 20% split *within that market*, using MAE and R2.
The table below is produced directly by the training run above.

In [2]:
# XGBoost needs to be installed BEFORE pricing_model.py is imported below - it checks
# for xgboost at import time (HAS_XGB), so installing it after import won't be picked up
# without restarting the kernel.
!pip install xgboost --quiet



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pricing_model import train_all_markets

results = train_all_markets()

Bangkok         (THB): MAE=   574.27  R2=-0.014  n=392
Cape Town       (ZAR): MAE=  1213.27  R2=0.223  n=395
Hong Kong       (HKD): MAE=   170.46  R2=0.560  n=392
Istanbul        (TRY): MAE=   142.37  R2=0.045  n=393
Mexico City     (MXN): MAE=   436.89  R2=0.335  n=392
New York        (USD): MAE=    48.45  R2=0.091  n=393
Paris           (EUR): MAE=    38.23  R2=0.289  n=394
Rio de Janeiro  (BRL): MAE=   372.03  R2=0.096  n=393
Rome            (EUR): MAE=    34.15  R2=0.276  n=392
Sydney          (AUD): MAE=    64.49  R2=0.650  n=392


In [4]:
eval_df = pd.DataFrame(results).T
eval_df

,mae,r2,n,currency
Bangkok,574.27,-0.014,392,THB
Cape Town,1213.27,0.223,395,ZAR
Hong Kong,170.46,0.56,392,HKD
Istanbul,142.37,0.045,393,TRY
Mexico City,436.89,0.335,392,MXN
New York,48.45,0.091,393,USD
Paris,38.23,0.289,394,EUR
Rio de Janeiro,372.03,0.096,393,BRL
Rome,34.15,0.276,392,EUR
Sydney,64.49,0.65,392,AUD
